# Backtesting Notebook (Refactor)

This notebook simulates the assignment rules using Close prices and 100-share lots.

Assumptions
- If a Buy Date is not a trading day, rebalance on the next available trading date.
- Daily NAV is computed for every calendar day using forward-filled prices.
- Fees accrue daily at 0.05% per year and are paid on Dec 31.


In [1]:
import pandas as pd
import numpy as np

## Configuration

In [2]:
DATA_PATH = "interday (intel-assignment).csv"
START_DATE = "2018-12-31"
END_DATE = "2021-12-31"

INITIAL_CASH = 100_000.0
LOT_SIZE = 100
ANNUAL_FEE_RATE = 0.0005  # 0.05%
DAILY_FEE_RATE = ANNUAL_FEE_RATE / 365
MAX_HOLDING_STOCKS_NUMBER = 4

## Rank Schedule

In [3]:
rank_schedule = {
    "2018-12-31": [2573042, 2572286, 1232815, 2572066],
    "2019-03-31": [2573125, 3695, 2572066, 2572067],
    "2019-06-30": [3695, 8893, 2572065, 2572067],
    "2019-09-30": [2573062, 2572286, 3695, 2572067],
    "2019-12-31": [2572856, 2572065, 851607, 2572765],
    "2020-03-31": [2573062, 851607, 497280, 2572066],
    "2020-06-30": [851607, 2572066, 497280, 2572067],
    "2020-09-30": [4572, 851607, 2572066, 1232815],
    "2020-12-31": [4572, 2572067, 2572765, 3695],
    "2021-03-31": [1232815, 2572067, 2572065, 7634],
    "2021-06-30": [2573085, 2572065, 2573062, 497280],
    "2021-09-30": [7634, 497280, 3695, 1232815],
}


## Load Prices

In [4]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

prices = df.pivot_table(
    index="timestamp",
    columns="jitta_stock_id",
    values="close",
).sort_index()


In [5]:
rank_ids = sorted({sid for stocks in rank_schedule.values() for sid in stocks})
df_prices = prices.reindex(columns=rank_ids)

## Schedule Mapping (Find Trading Dates)

In [6]:
trading_dates = df_prices.index

def map_to_trading_date(raw_date):
    if raw_date in trading_dates:
        return raw_date
    pos = trading_dates.searchsorted(raw_date)
    if pos >= len(trading_dates):
        raise ValueError(f"No trading date on or after {raw_date}")
    return trading_dates[pos]

effective_schedule = {
    map_to_trading_date(pd.to_datetime(d)): stocks
    for d, stocks in rank_schedule.items()
}


## Trade Log Helpers

In [7]:
tx_rows = []

def log_tx(date, action, stock_id, shares, price, amount, cash_after, note):
    tx_rows.append({
        "date": date,
        "action": action,
        "stock_id": stock_id,
        "shares": shares,
        "price": price,
        "amount": amount,
        "cash_after": cash_after,
        "note": note,
    })


## Rebalance Logic

In [8]:
def rebalance(date, target_stocks, holdings, cash, price_row, lot_size=LOT_SIZE):
    total_value = cash
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        total_value += shares * price_row[sid]

    target_value = total_value / len(target_stocks)

    desired = {}
    for sid in target_stocks:
        px = price_row[sid]
        lots = np.floor(target_value / (px * lot_size))
        desired[sid] = int(lots * lot_size)

    # SELL first
    all_sids = sorted(set(holdings.keys()) | set(target_stocks))
    for sid in all_sids:
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur > tgt:
            shares = cur - tgt
            proceeds = shares * price_row[sid]
            cash += proceeds
            holdings[sid] = tgt
            log_tx(date, "SELL", sid, shares, price_row[sid], proceeds, cash, "rebalance")

    # BUY after
    for sid in target_stocks:
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur < tgt:
            shares = tgt - cur
            cost = shares * price_row[sid]
            if cost > cash + 1e-9:
                max_lots = int(cash // (price_row[sid] * lot_size))
                shares = max_lots * lot_size
                if shares == 0:
                    continue
                cost = shares * price_row[sid]
                tgt = cur + shares
            cash -= cost
            holdings[sid] = tgt
            log_tx(date, "BUY", sid, shares, price_row[sid], -cost, cash, "rebalance")

    return holdings, cash


## Run Simulation

In [9]:
dates = pd.date_range(START_DATE, END_DATE, freq="D")
daily_prices = df_prices.reindex(dates).ffill()

stock_cols = [f"stock{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]
lot_cols = [f"lot{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]
value_cols = [f"value{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]

holdings = {}
cash = INITIAL_CASH
accrued_fee = 0.0
current_order = []

nav_rows = []

for date in dates:
    is_rebalance = False
    is_payfee = False

    price_row = daily_prices.loc[date]

    if date in effective_schedule:
        current_order = effective_schedule[date]
        holdings, cash = rebalance(date, current_order, holdings, cash, price_row)
        is_rebalance = True

    held = [sid for sid in current_order if holdings.get(sid, 0) > 0]
    held = held[:MAX_HOLDING_STOCKS_NUMBER]
    held = held + [None] * (MAX_HOLDING_STOCKS_NUMBER - len(held))

    stock_value_total = 0.0
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        stock_value_total += shares * price_row[sid]

    asset_value = stock_value_total + cash - accrued_fee
    daily_fee = asset_value * DAILY_FEE_RATE
    accrued_fee += daily_fee

    if date.month == 12 and date.day == 31 and accrued_fee > 0:
        cash -= accrued_fee
        log_tx(date, "FEE_PAYMENT", None, None, None, -accrued_fee, cash, "year-end")
        accrued_fee = 0.0
        is_payfee = True

    lots = []
    values = []
    for sid in held:
        if sid is None:
            lots.append(0)
            values.append(0.0)
        else:
            shares = holdings[sid]
            lots.append(shares // LOT_SIZE)
            values.append(shares * price_row[sid])

    nav = stock_value_total + cash - accrued_fee

    nav_rows.append({
        "date": date,
        **{stock_cols[i]: held[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        **{lot_cols[i]: lots[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        **{value_cols[i]: values[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        "stock_value_total": stock_value_total,
        "cash": cash,
        "accrued_fee": accrued_fee,
        "nav": nav,
        "is_rebalance": is_rebalance,
        "is_payfee": is_payfee,
    })

df_nav = pd.DataFrame(nav_rows).set_index("date").sort_index()
df_trade_log = pd.DataFrame(tx_rows).sort_values("date").reset_index(drop=True)


In [ ]:
df_nav.head()

,stock1,stock2,stock3,stock4,lot1,lot2,lot3,lot4,value1,value2,value3,value4,stock_value_total,cash,accrued_fee,nav,is_rebalance,is_payfee
date,,,,,,,,,,,,,,,,,,
2018-12-31,2573042,2572286,1232815,2572066,15,6,2,8,24666.585,23506.4322,16987.8840,23886.1096,89047.0108,10952.852214,0.000000,99999.863014,True,True
2019-01-01,2573042,2572286,1232815,2572066,15,6,2,8,24666.585,23506.4322,16987.8840,23886.1096,89047.0108,10952.852214,0.136986,99999.726028,False,False
2019-01-02,2573042,2572286,1232815,2572066,15,6,2,8,24502.710,23524.3440,17112.4516,24138.7888,89278.2944,10952.852214,0.274289,100230.872325,False,False
2019-01-03,2573042,2572286,1232815,2572066,15,6,2,8,23891.271,22640.6886,16084.7702,23592.6856,86209.4154,10952.852214,0.407387,97161.860226,False,False
2019-01-04,2573042,2572286,1232815,2572066,15,6,2,8,24877.062,23769.1404,16834.1212,24849.4504,90329.7740,10952.852214,0.546130,101282.080083,False,False


In [ ]:
df_trade_log.head()

,date,action,stock_id,shares,price,amount,cash_after,note
0,2018-12-31,BUY,2573042.0,1500.0,16.444390,-24666.585000,75333.415000,rebalance
1,2018-12-31,BUY,2572286.0,600.0,39.177387,-23506.432200,51826.982800,rebalance
2,2018-12-31,BUY,1232815.0,200.0,84.939420,-16987.884000,34839.098800,rebalance
3,2018-12-31,BUY,2572066.0,800.0,29.857637,-23886.109600,10952.989200,rebalance
4,2018-12-31,FEE_PAYMENT,NaN,NaN,NaN,-0.136986,10952.852214,year-end


## Export (Optional)

In [12]:
df_nav.to_csv("nav.csv", index=True)
df_trade_log.to_csv("trade_log.csv", index=True)


## Return Metrics

In [ ]:
def calc_return_metrics(df_nav):
    nav_start = df_nav["nav"].iloc[0]
    nav_end = df_nav["nav"].iloc[-1]

    total_return = (nav_end - nav_start) / nav_start
    cagr = (nav_end / nav_start) ** (1 / years) - 1

    return {
        "Total_Return": total_return,
        "CAGR": cagr,
    }

metrics = calc_return_metrics(df_nav)
pd.DataFrame(metrics, index=["value"]).T


,value
Total_Return,1.600900
CAGR,0.374828
Years,3.002740
